## Загрузка и подготовка данных

In [43]:
import torch
from torch import nn
import torch.optim as optim

import pickle
from tqdm import tqdm

import numpy as np
import matplotlib.pyplot as plt

import torch.nn.functional as F

In [44]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader

In [45]:
# Определение преобразования
fashion_transformer = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(0.13, 0.31)])

# Загрузка данных
train_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=fashion_transformer
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data',
    train=False,
    download=True,
    transform=fashion_transformer
)

# Классы fashionMNIST
fashion_classes = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print(f"Размер тренировочного набора: {len(train_dataset)}")
print(f"Размер тестового набора: {len(test_dataset)}")

Размер тренировочного набора: 60000
Размер тестового набора: 10000


## Создание AlexNet

In [46]:
# Определение устройства
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

    print(device)

cpu


In [51]:
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        self.layers = nn.Sequential(
             # Первый сверточный блок - адаптирован для 28x28
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),  # 28x28 -> 28x28
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 28x28 -> 14x14

            # Второй сверточный блок
            nn.Conv2d(64, 192, kernel_size=3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14 -> 7x7

            # Третий сверточный блок
            nn.Conv2d(192, 384, kernel_size=3, padding=1),  # 7x7 -> 7x7
            nn.ReLU(inplace=True),

            # Четвертый сверточный блок
            nn.Conv2d(384, 256, kernel_size=3, padding=1),  # 7x7 -> 7x7
            nn.ReLU(inplace=True),

            # Пятый сверточный блок
            nn.Conv2d(256, 256, kernel_size=3, padding=1),  # 7x7 -> 7x7
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 7x7 -> 3x3 (округление вниз)
        )

         # Классификатор
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 3 * 3, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    # прогоняем вектор x через все слои
    def forward(self, x):
        x = self.layers(x)  # [batch, 256, 3, 3]
        x = torch.flatten(x, 1)  # [batch, 256 * 3 * 3]
        x = self.classifier(x)  # [batch, num_classes]
        return x

# Инициализация модели
model = AlexNet(num_classes=10).to(device)
print(model)

AlexNet(
  (layers): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=2304, out_features=4096, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.5, inpl

In [52]:
# Создание DataLoader
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Функция обучение

In [53]:
def train(model, train_loader, device, num_epochs):

    # результат функции train
    train_losses = []
    train_accuracies = []

    optimizer = torch.optim.SGD(model.parameters(), lr=0.01) # алгоритм, который обновляет веса модели, чтобы минимизировать функцию потерь

    for epoch in range(num_epochs):
        model.train()  # Режим обучения
        total_loss = 0
        total = 0
        correct = 0

        for batch_idx, (data, target) in enumerate(train_loader):

            # Перемещаем данные на устройство
            data, target = data.to(device), target.to(device)

            # Обнуляем градиенты ПЕРЕД прямым проходом
            optimizer.zero_grad()

            # Прямой проход
            outputs = model(data)
            loss = F.cross_entropy(outputs, target)
            _, predicted = outputs.max(1)

            # Обратный проход
            loss.backward()        # Вычисляем градиенты
            optimizer.step()       # Обновляем веса

            # Статистика
            total_loss += loss.item()
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

        epoch_loss = total_loss/len(train_loader)
        epoch_acc = 100. * correct / total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)

        if (epoch+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')


    return train_losses, train_accuracies


## Запуск модели

In [ ]:
# Параметры обучения
num_epochs = 10

# Обучение
train_losses, train_accuracies = train(model, train_loader, device, num_epochs)

## Оценка

In [ ]:
model.eval()  # Режим оценки
correct = 0
total = 0

with torch.no_grad():  # Отключаем вычисление градиентов
    for data, target in test_loader:
        data = data.float()

        outputs = model(data)
        _, predicted = torch.max(outputs.data, dim=1)  # torch.max(tensor, dim) -> (values, indices); dim=1 - максимум по второму измерению(по классам)
        total += target.size(0) # возвращает размер батча
        correct += (predicted == target).sum()

print(f'Accuracy: {100 * correct / total:.2f}%')